In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
import seaborn as sns
from tensorflow.keras import layers
import zipfile
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/homer_bart_1.zip'
zip_object = zipfile.ZipFile(file = path, mode = 'r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
diretorio = '/content/homer_bart_1'
arquivos = [os.path.join(diretorio, f) for f in sorted(os.listdir(diretorio))]

batch_size = 32
img_height, img_width = 180, 180

In [ ]:
imagens = []
classes = []

In [ ]:
for imagem_caminho in arquivos:
  try:
    imagem = cv2.imread(imagem_caminho)
    (H, W) = imagem.shape[:2]
  except:
    continue

  imagem = cv2.resize(imagem, (img_height, img_width))
  imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2GRAY)

  imagem = imagem.ravel()

  imagens.append(imagem)
  nome_imagem = os.path.basename(os.path.normpath(imagem_caminho))

  if nome_imagem.startswith('b'):
    classe = 0
  else:
    classe = 1

  classes.append(classe)

In [ ]:
previsores = np.asarray(imagens)
classes = np.asarray(classes)

In [ ]:
previsores.shape

(269, 32400)

In [ ]:
classes.shape

(269,)

In [ ]:
sns.countplot(classes);

In [ ]:
np.unique(classes, return_counts=True)

(array([0, 1]), array([160, 109]))

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
previsores = scaler.fit_transform(previsores)

In [ ]:
previsores

array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [0.99479167, 0.99479167, 0.99479167, ..., 0.996     , 0.995671  ,
        0.99479167],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [0.99479167, 1.        , 1.        , ..., 1.        , 1.        ,
        0.984375  ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ]])

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(previsores, classes, test_size=0.2, random_state=0)

In [ ]:
x_train.shape[1]

32400

In [ ]:
model = Sequential()

model.add(tf.keras.layers.Dense(5120, activation='relu', input_shape=(32400, )))
model.add(tf.keras.layers.Dense(5120, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 5120)              165893120 
                                                                 
 dense_1 (Dense)             (None, 5120)              26219520  
                                                                 
 dense_2 (Dense)             (None, 1)                 5121      
                                                                 
Total params: 192117761 (732.87 MB)
Trainable params: 192117761 (732.87 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
model.compile(loss='binary_crossentropy', metrics='accuracy', optimizer='adam')
historico = model.fit(x_train, y_train, batch_size=64,epochs=50, validation_data=(x_test, y_test))

In [ ]:
historico.history.keys()

dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])

In [ ]:
plt.plot(historico.history['loss']);
plt.plot(historico.history['val_loss']);

In [ ]:
plt.plot(historico.history['accuracy']);
plt.plot(historico.history['val_accuracy']);

In [ ]:
pred = model.predict(x_test)
pred = (pred > 0.5)

2/2 [==============================] - 0s 10ms/step


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, pred)


sns.heatmap(cm, annot=True)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.65      0.83      0.73        29
           1       0.71      0.48      0.57        25

    accuracy                           0.67        54
   macro avg       0.68      0.65      0.65        54
weighted avg       0.68      0.67      0.66        54



In [ ]:
modelo = model.to_json()
with open('modelo.json', 'w') as json_file:
  json_file.write(modelo)

In [ ]:
from keras.models import save_model
model_save = save_model(model, 'weights.hdf5')

<ipython-input-35-1310051c93f8>:2: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  model_save = save_model(model, 'weights.hdf5')


In [ ]:
with open('modelo.json') as json_file:
  json_saved_model = json_file.read()
json_saved_model

'{"class_name": "Sequential", "config": {"name": "sequential", "layers": [{"module": "keras.layers", "class_name": "InputLayer", "config": {"batch_input_shape": [null, 32400], "dtype": "float32", "sparse": false, "ragged": false, "name": "dense_input"}, "registered_name": null}, {"module": "keras.layers", "class_name": "Dense", "config": {"name": "dense", "trainable": true, "dtype": "float32", "batch_input_shape": [null, 32400], "units": 5120, "activation": "relu", "use_bias": true, "kernel_initializer": {"module": "keras.initializers", "class_name": "GlorotUniform", "config": {"seed": null}, "registered_name": null}, "bias_initializer": {"module": "keras.initializers", "class_name": "Zeros", "config": {}, "registered_name": null}, "kernel_regularizer": null, "bias_regularizer": null, "activity_regularizer": null, "kernel_constraint": null, "bias_constraint": null}, "registered_name": null, "build_config": {"input_shape": [null, 32400]}}, {"module": "keras.layers", "class_name": "Dense

In [ ]:
model_loaded = tf.keras.models.model_from_json(json_saved_model)
model_loaded.load_weights('/content/weights.hdf5')
model_loaded.compile(loss = 'binary_crossentropy',
                     optimizer = 'adam',
                     metrics = ['accuracy'])

In [ ]:
model_loaded.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 5120)              165893120 
                                                                 
 dense_1 (Dense)             (None, 5120)              26219520  
                                                                 
 dense_2 (Dense)             (None, 1)                 5121      
                                                                 
Total params: 192117761 (732.87 MB)
Trainable params: 192117761 (732.87 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
imagem_test = x_test[50]
imagem_test =  scaler.inverse_transform(imagem_test.reshape(1, -1))

In [ ]:
cv2_imshow(imagem_test.reshape(180, 180))

In [ ]:
if model_loaded.predict(imagem_test)[0][0] < 0.5:
  print('Bart')
else:
  print('Hommer')

1/1 [==============================] - 0s 37ms/step
Bart
